# Решение задачи по детекции ботов

## Вспомогатльные функции

In [463]:
import polars as pl

In [464]:
def events_in_window(events: pl.DataFrame, meta: pl.DataFrame) -> pl.DataFrame:
    ev = events.join(meta, on='cookie_id')

    return ev.filter(
        (pl.col('event_ts') >= pl.col('window_start_ts')) &
        (pl.col('event_ts') < pl.col('window_end_ts'))
    )

In [465]:
keep_cols = ['cookie_id', 'window_start_ts', 'window_end_ts']

In [466]:
# Выравнивние фичей по мета-таблице (train/test)
def align_to_meta(meta: pl.DataFrame, features: pl.DataFrame, keep_meta_cols: bool = False) -> pl.DataFrame:
    aligned = (
        meta
        .select(keep_cols)
        .join(features, on='cookie_id', how='left')
        .fill_null(0)
    )

    if keep_meta_cols:
        return aligned
    else:
        return aligned.drop(keep_cols)

In [467]:
def add_event_deltas(events: pl.DataFrame) -> pl.DataFrame:
    return (
        events
        .sort(['cookie_id', 'event_ts'])
        .with_columns(
            pl.col('event_ts')
            .diff()
            .dt.total_seconds()
            .over('cookie_id')
            .alias('delta_sec')
        )
    )

In [468]:
# Session gap был выявлен в ходе исследования
def add_session_id(event: pl.DataFrame, session_gap_sec: int = 180) -> pl.DataFrame:
    return (
        add_event_deltas(event)
        .with_columns(
            (pl.col('delta_sec') > session_gap_sec)
            .fill_null(True)
            .cum_sum()
            .over('cookie_id')
            .alias('session_id')
        )
    )

## Фичи

### Базовые

In [469]:
def basic_features(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    features = (
        events
        .group_by('cookie_id')
        .agg(
            pl.len().alias('n_events'),
            pl.col('item_id').n_unique().alias('n_unique_items'),
            pl.col('item_category').n_unique().alias('n_unique_categories'),
            pl.col('item_location').n_unique().alias('n_unique_locations')
        )
    )

    return align_to_meta(meta, features, True)

### Временные

In [470]:
def temporal_basic_features(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    features = (
        add_event_deltas(events)
        .group_by("cookie_id")
        .agg(
            pl.col("delta_sec").mean().alias("delta_mean"),
            pl.col("delta_sec").median().alias("delta_median"),
            pl.col("delta_sec").std().alias("delta_std"),
            pl.col("delta_sec").min().alias("delta_min"),
            pl.col("delta_sec").max().alias("delta_max"),
        )
        .with_columns(
            (pl.col("delta_std") / pl.col("delta_mean")).alias("delta_cv")
        )
    )

    return align_to_meta(meta, features)

In [471]:
def temporal_fast_events(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    features = (
        add_event_deltas(events)
        .group_by("cookie_id")
        .agg(
            (pl.col("delta_sec") < 10).mean().alias("pct_delta_lt_10s"),
            (pl.col("delta_sec") < 30).mean().alias("pct_delta_lt_30s"),
        )
    )

    return align_to_meta(meta, features)

In [472]:
def temporal_sessions(X: tuple[pl.DataFrame, pl.DataFrame], session_gap_sec: int = 180) -> pl.DataFrame:
    meta, events = X

    features = (
        add_session_id(events, session_gap_sec)
        .group_by(["cookie_id", "session_id"])
        .agg(
            pl.len().alias("events_in_session"),
            (pl.col("event_ts").max() - pl.col("event_ts").min())
            .dt.total_seconds()
            .alias("session_duration_sec"),
        )
        .group_by("cookie_id")
        .agg(
            pl.len().alias("n_sessions"),
            pl.col("events_in_session").mean().alias("avg_events_per_session"),
            pl.col("events_in_session").max().alias("max_events_per_session"),
            pl.col("session_duration_sec").mean().alias("avg_session_duration_sec"),
            pl.col("session_duration_sec").max().alias("max_session_duration_sec"),
        )
    )

    return align_to_meta(meta, features)

### Паттерны

In [473]:
def search_page_pattern_features(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    features = (
        events
        .filter(pl.col('search_page').is_not_null())
        .sort(['cookie_id', 'event_ts'])
        .with_columns(
            (pl.col('search_page').diff().over('cookie_id') == 1)
            .fill_null(False)
            .alias('is_sequential_page')
        )
        .group_by('cookie_id')
        .agg(
            pl.col('is_sequential_page').mean().alias('sequential_page_ratio'),
        )
    )

    return align_to_meta(meta, features)

In [474]:
def event_type_mix_features(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    features = (
        events
        .group_by('cookie_id')
        .agg(
            (pl.col('event_name') == 'item_view').mean().alias('ratio_item_view'),
            (pl.col('event_name') == 'search_results_view').mean().alias('ratio_search_view'),
        )
    )

    return align_to_meta(meta, features)

In [475]:
def pointer_features(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    features = (
        events
        .group_by('cookie_id')
        .agg(
            pl.col('pointer_x').is_not_null().mean().alias('pointer_present_ratio'),
            pl.col('pointer_y').drop_nulls().std().alias('pointer_y_std'),
        )
    )

    return align_to_meta(meta, features)

### Cross cookie

In [476]:
def cross_cookie_item_overlap(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    n_total_cookies = events.get_column('cookie_id').n_unique()

    item_popularity = (
        events
        .filter(pl.col('item_id').is_not_null())
        .group_by('item_id')
        .agg(pl.col('cookie_id').n_unique().alias('n_cookies_viewed_item'))
        .with_columns(
            (pl.col('n_cookies_viewed_item') / n_total_cookies).alias('item_overlap_share')
        )
    )

    ev = events.join(item_popularity, on='item_id', how='left')

    # относительный порог вместо абсолютного >5
    q90 = item_popularity.get_column('item_overlap_share').quantile(0.9)

    features = (
        ev
        .group_by('cookie_id')
        .agg(
            pl.col('item_overlap_share').mean().alias('avg_item_overlap_share'),
            pl.col('item_overlap_share').max().alias('max_item_overlap_share'),
            (pl.col('item_overlap_share') > q90).mean().alias('pct_high_overlap_items'),
        )
    )

    return align_to_meta(meta, features)

In [477]:
def item_sequence_monotonicity(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    features = (
        events
        .filter(pl.col('item_id').is_not_null())
        .sort(['cookie_id', 'event_ts'])
        .with_columns(
            pl.col('item_id').diff().over('cookie_id').alias('item_id_diff')
        )
        .group_by('cookie_id')
        .agg(
            (pl.col('item_id_diff') > 0).mean().alias('pct_increasing_item_id'),
            (pl.col('item_id_diff') < 0).mean().alias('pct_decreasing_item_id'),
        )
        .with_columns(
            pl.max_horizontal('pct_increasing_item_id', 'pct_decreasing_item_id')
            .alias('item_id_monotonicity')
        )
    )

    return align_to_meta(meta, features)

In [478]:
def query_overlap_features(X: tuple[pl.DataFrame, pl.DataFrame]) -> pl.DataFrame:
    meta, events = X

    pool_size = (
        events
        .group_by('window_start_ts')
        .agg(pl.col('cookie_id').n_unique().alias('n_cookies_in_window'))
    )

    query_popularity = (
        events
        .filter(pl.col('search_query').is_not_null())
        .group_by(['window_start_ts', 'search_query'])
        .agg(pl.col('cookie_id').n_unique().alias('n_cookies_same_query'))
        .join(pool_size, on='window_start_ts')
        .with_columns(
            (pl.col('n_cookies_same_query') / pl.col('n_cookies_in_window'))
            .alias('query_overlap_share')
        )
    )

    ev = events.join(
        query_popularity.select(['window_start_ts', 'search_query', 'query_overlap_share']),
        on=['window_start_ts', 'search_query'],
        how='left'
    )

    features = (
        ev
        .group_by('cookie_id')
        .agg(
            pl.col('query_overlap_share').mean().alias('avg_query_overlap_share'),
            pl.col('query_overlap_share').max().alias('max_query_overlap_share'),
        )
    )

    return align_to_meta(meta, features)

## Pipeline

In [479]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import FeatureUnion
from catboost import CatBoostClassifier
from metric import precision_at_recall

In [480]:
train = pl.read_csv('data/train.csv', try_parse_dates=True)
test = pl.read_csv('data/test.csv', try_parse_dates=True)
events = pl.read_csv('data/events.csv.gz', try_parse_dates=True)

In [481]:
ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

# Убираем дубликаты событий
ev_tr = ev_tr.unique()
ev_te = ev_te.unique()

# Нормализуем platform
ev_tr = ev_tr.with_columns(pl.col('platform').str.to_lowercase())
ev_te = ev_te.with_columns(pl.col('platform').str.to_lowercase())

In [482]:
features_union = FeatureUnion([
    ('basic', FunctionTransformer(basic_features)),
    ('temporal_basic', FunctionTransformer(temporal_basic_features)),
    ('temporal_fast_events', FunctionTransformer(temporal_fast_events)),
    ('temporal_sessions', FunctionTransformer(temporal_sessions)),
    ('search_page_pattern_features', FunctionTransformer(search_page_pattern_features)),
    ('event_type_mix', FunctionTransformer(event_type_mix_features)),
    ('pointer', FunctionTransformer(pointer_features)),
    ('cross_cookie_item_overlap', FunctionTransformer(cross_cookie_item_overlap)),
    ('item_sequence_monotonicity', FunctionTransformer(item_sequence_monotonicity)),
    ('query_overlap_features', FunctionTransformer(query_overlap_features))
    
]).set_output(transform='polars')

In [483]:
Xtr = features_union.transform((train, ev_tr))
Xte = features_union.transform((test, ev_te))

print(Xtr.shape, Xte.shape)

(11091, 33) (4909, 33)


In [484]:
is_valid = pl.col('basic__window_start_ts') >= pl.datetime(2026, 4, 15)
is_valid_y = pl.col('window_start_ts') >= pl.datetime(2026, 4, 15)
drop_cols = ['basic__cookie_id','basic__window_start_ts', 'basic__window_end_ts']

In [485]:
X_train = Xtr.filter(~is_valid).drop(drop_cols)
X_valid = Xtr.filter(is_valid).drop(drop_cols)

print(X_train.shape, X_valid.shape)

y_train = train.filter(~is_valid_y).get_column('target')
y_valid = train.filter(is_valid_y).get_column('target')

print(y_train.shape, y_valid.shape)

(7599, 30) (3492, 30)
(7599,) (3492,)


In [486]:
model = CatBoostClassifier(
    iterations=1500,
    loss_function='Logloss',
    random_state=512,
    verbose=False,
)

model.fit(X_train, y_train)

p_va = model.predict_proba(X_valid)[:, 1]

round(precision_at_recall(y_valid, p_va), 4)

0.7363

## Submit

In [487]:
model.fit(Xtr.drop(drop_cols), train.get_column('target'))

sub = pl.DataFrame({
    'cookie_id': Xte.get_column('basic__cookie_id'),
    'score': model.predict_proba(Xte.drop(drop_cols))[:, 1]
})

assert len(sub) == len(test) and sub.get_column('score').is_between(0, 1).all()

sub.write_csv('submission.csv')
sub.head()

cookie_id,score
str,f64
"""ck_315fb710a0e371e7""",0.002983
"""ck_a76ee3b3e3e522fd""",0.09685
"""ck_94c9a4d382689e82""",0.186122
"""ck_8eaf9509ad9462a0""",0.003388
"""ck_9a88a5a989cb5bc6""",0.026492
